In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")

# Reto 1: Olivetti Faces
Clasificación de 40 rostros usando una red fully-connected en Keras 3.

In [ ]:
# Carga de datos
faces = fetch_olivetti_faces(shuffle=True, random_state=SEED)
X_oli = faces.images
y_oli = faces.target

# Split estratificado
X_train_oli, X_test_oli, y_train_oli, y_test_oli = train_test_split(
    X_oli, y_oli, test_size=0.20, random_state=SEED, stratify=y_oli
)

print("X_train_oli:", X_train_oli.shape)
print("X_test_oli :", X_test_oli.shape)
print("Número de clases:", len(np.unique(y_oli)))

**Pregunta 1:** Cuando apliquemos `Flatten()` a imágenes de 64x64, ¿cuántos valores tendrá cada ejemplo?
**Respuesta:** Multiplicamos 64x64. El resultado es 4096. Por lo tanto, cada ejemplo se convertirá en un vector de 4096 valores.

In [ ]:
# Arquitectura
model_olivetti = keras.Sequential([
    keras.Input(shape=(64, 64)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(40)
])

# Configuración
lr_rate = 0.001
model_olivetti.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# Entrenamiento
history_oli = model_olivetti.fit(
    X_train_oli, y_train_oli,
    epochs=30, batch_size=32,
    validation_data=(X_test_oli, y_test_oli),
    shuffle=True, verbose=0 # Verbose 0 para no llenar la pantalla
)

# Evaluación
test_loss_oli, test_accuracy_oli = model_olivetti.evaluate(X_test_oli, y_test_oli, batch_size=32, verbose=0)
print(f"Accuracy en Test (Olivetti): {test_accuracy_oli * 100:.2f}%")

* **P2. ¿Por qué Flatten()?** Transforma la matriz 2D en un vector 1D, requisito para las capas Dense.
* **P3. ¿Por qué 40 neuronas al final?** Porque hay exactamente 40 personas (clases) en el dataset.
* **P4. ¿Función de ReLU?** Introduce no linealidad, permitiendo aprender representaciones complejas.
* **P5. ¿Por qué from_logits=True?** Porque la última capa no tiene Softmax, emitiendo valores crudos (logits).
* **P6. ¿Qué es una epoch?** Un ciclo donde la red procesa todo el dataset de entrenamiento una vez.
* **P7. ¿Qué es batch_size=32?** El modelo agrupa 32 imágenes para el cálculo de gradientes antes de actualizar los pesos.
* **P8. ¿Hay overfitting?** Sí. El accuracy de entrenamiento roza el 100% mientras la validación se estanca (las curvas divergen).
* **P9. ¿Es mucha o poca data?** Extremadamente poca. Con más de 260,000 parámetros y solo 320 imágenes para entrenar, el modelo memoriza el ruido.
* **P10. ¿Por qué Softmax suma 1?** Porque normaliza los logits en una distribución de probabilidad para categorías mutuamente excluyentes.

In [ ]:
# Tomamos una imagen de prueba
image_batch_oli = np.expand_dims(X_test_oli[0], axis=0)
logits_oli = model_olivetti(image_batch_oli, training=False)

# Logits a Probabilidades
probabilities_oli = keras.ops.softmax(logits_oli)
prediction_oli = keras.ops.argmax(probabilities_oli, axis=1)

print("Clase real     :", y_test_oli[0])
print("Clase predicha :", int(keras.ops.convert_to_numpy(prediction_oli)[0]))

---
# Reto 2: CIFAR-10 Grayscale
Clasificación de 10 clases usando imágenes de 32x32 convertidas a escala de grises.

In [ ]:
(X_train_cifar, y_train_cifar), (X_test_cifar, y_test_cifar) = keras.datasets.cifar10.load_data()
y_train_cifar = y_train_cifar.squeeze()
y_test_cifar = y_test_cifar.squeeze()
class_names_cifar = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

# Convertir a escala de grises
X_train_cifar = 0.299 * X_train_cifar[..., 0] + 0.587 * X_train_cifar[..., 1] + 0.114 * X_train_cifar[..., 2]
X_test_cifar = 0.299 * X_test_cifar[..., 0] + 0.587 * X_test_cifar[..., 1] + 0.114 * X_test_cifar[..., 2]

# TODO 1: Normalización
X_train_cifar = X_train_cifar.astype("float32") / 255.0
X_test_cifar  = X_test_cifar.astype("float32") / 255.0

print("X_train grayscale normalizado:", X_train_cifar.shape)

**Pregunta 1:** ¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?
**Respuesta:** Cada imagen de 32x32 tendrá 1024 valores.

In [ ]:
# Arquitectura
model_cifar = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

# Configuración
model_cifar.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# Entrenamiento
history_cifar = model_cifar.fit(
    X_train_cifar, y_train_cifar,
    epochs=10, batch_size=64,
    validation_data=(X_test_cifar, y_test_cifar),
    shuffle=True, verbose=1
)

# Evaluación
test_loss_cifar, test_accuracy_cifar = model_cifar.evaluate(X_test_cifar, y_test_cifar, batch_size=64, verbose=0)
print(f"\nAccuracy en Test (CIFAR-10): {test_accuracy_cifar * 100:.2f}%")

In [ ]:
# Predicción
image_batch_cif = np.expand_dims(X_test_cifar[0], axis=0)
logits_cif = model_cifar(image_batch_cif, training=False)

probabilities_cif = keras.ops.softmax(logits_cif)
prediction_cif = keras.ops.argmax(probabilities_cif, axis=1)
pred_idx = int(keras.ops.convert_to_numpy(prediction_cif)[0])

print("Clase real     :", class_names_cifar[y_test_cifar[0]])
print("Clase predicha :", class_names_cifar[pred_idx])

**Respuestas a Preguntas Finales:**
* **1. Accuracy final:** Oscila alrededor del 40%-45%.
* **2. Clases más difíciles:** Las biológicas (perros vs gatos) o siluetas difusas sobre fondos grises, ya que se perdió el color.
* **3. Pérdida de info con Flatten():** Se destruyen las correlaciones espaciales en 2D (texturas, formas, cercanía entre píxeles).
* **4. Arquitectura ideal:** Redes Neuronales Convolucionales (CNNs), que aplican filtros locales preservando la matriz 2D original.